In [20]:
import pandas as pd
import numpy as np

dfAnime = pd.read_csv('/home/jsnnb/anime-recommender/data/raw/anime.csv')
dfRating = pd.read_csv('/home/jsnnb/anime-recommender/data/raw/rating.csv')

In [21]:

dfAnime.shape


(12294, 7)

In [22]:
dfAnime.head(5)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [23]:
dfRating.shape

(7813737, 3)

In [24]:
dfRating.head(5)

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


In [25]:
dfAnime.isnull().sum()


anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [26]:
dfRating.isnull().sum()

user_id     0
anime_id    0
rating      0
dtype: int64

In [27]:
dfAnime = dfAnime.dropna(subset=['genre', 'rating'])
dfAnime['type'] = dfAnime['type'].fillna('Unknown')

dfAnime.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [28]:
dfAnime.duplicated().sum()

np.int64(0)

In [29]:
dfAnime[dfAnime['name'].str.contains('&#', na=False)][['anime_id', 'name']].head(10)

,anime_id,name
4,9969,Gintama&#039;
9,15417,Gintama&#039;: Enchousen
187,28675,Kyoukai no Kanata Movie: I&#039;ll Be Here - M...
194,10153,Mahou Shoujo Lyrical Nanoha: The Movie 2nd A&#...
359,4901,Black Lagoon: Roberta&#039;s Blood Trail
387,1023,Wolf&#039;s Rain OVA
416,77,Mahou Shoujo Lyrical Nanoha A&#039;s
553,10521,Working&#039;!!
641,202,Wolf&#039;s Rain
651,30745,Kyoukai no Kanata Movie: I&#039;ll Be Here - K...


In [30]:
import re

padroes = dfAnime[dfAnime['name'].str.contains('&#', na=False)]['name']
codigos = padroes.str.findall(r'&#\d+;').explode().unique()
print(codigos)

<ArrowStringArray>
['&#039;']
Length: 1, dtype: str


In [32]:
import html
dfAnime['name'] = dfAnime['name'].apply(html.unescape)
dfAnime[dfAnime['name'].str.contains('&#', na=False)]

,anime_id,name,genre,type,episodes,rating,members


In [33]:
print(dfRating.duplicated().sum())
print(dfRating['rating'].value_counts().sort_index())

1
rating
-1     1476496
 1       16649
 2       23150
 3       41453
 4      104291
 5      282806
 6      637775
 7     1375287
 8     1646019
 9     1254096
 10     955715
Name: count, dtype: int64


In [34]:
dfRating = dfRating[dfRating['rating'] != -1]
print(dfRating.shape)
print(dfRating['rating'].value_counts().sort_index())

(6337241, 3)
rating
1       16649
2       23150
3       41453
4      104291
5      282806
6      637775
7     1375287
8     1646019
9     1254096
10     955715
Name: count, dtype: int64


In [36]:
dfRating[dfRating.duplicated(keep=False)]

,user_id,anime_id,rating
4499258,42653,16498,8
4499316,42653,16498,8


In [37]:
dfRating = dfRating.drop_duplicates()
print(dfRating.duplicated().sum())
print(dfRating.shape)

0
(6337240, 3)


In [39]:
contagem = dfRating['anime_id'].value_counts()
print(contagem.describe())

count     9927.000000
mean       638.384205
std       1795.864158
min          1.000000
25%          9.000000
50%         57.000000
75%        395.000000
max      34226.000000
Name: count, dtype: float64


In [45]:
animes_validos = contagem[contagem >= 250].index
print(len(animes_validos))

3004


In [46]:
dfRating = dfRating[dfRating['anime_id'].isin(animes_validos)]
print(dfRating.shape)

(5192793, 3)


In [47]:
generos = dfAnime['genre'].str.split(', ').explode()
print(generos.value_counts().head(10))

genre
Comedy           4575
Action           2768
Adventure        2316
Fantasy          2242
Sci-Fi           2036
Drama            1977
Shounen          1684
Kids             1598
Romance          1437
Slice of Life    1204
Name: count, dtype: int64


In [48]:
print(dfRating['rating'].describe())

count    5.192793e+06
mean     7.942013e+00
std      1.516799e+00
min      1.000000e+00
25%      7.000000e+00
50%      8.000000e+00
75%      9.000000e+00
max      1.000000e+01
Name: rating, dtype: float64


In [51]:
dfAnime.to_csv('/home/jsnnb/anime-recommender/data/processed/anime_clean.csv', index=False)
dfRating.to_csv('/home/jsnnb/anime-recommender/data/processed/rating_clean.csv', index=False)